# Wake-Word Dataset Generator for Prosthetic Hand Control  
**Author:** Joaquín Cerdá-Boluda  
**Version:** 1.0  
**Last update:** 10/11/2025  
[GitHub Repository](https://github.com/ximocerda/ProstheticHand-VoiceCommands) ---

## Introduction

This notebook provides a complete and reproducible pipeline for generating a **synthetic voice-command dataset** specifically tailored for keyword spotting (KWS) and voice-driven control of a prosthetic hand.

Instead of relying on proprietary APIs or large-scale human recordings, this workflow is built entirely on **open-source components** and runs seamlessly in Google Colab. Speech synthesis is performed using the HuggingFace MMS-TTS English model, and additional variability is introduced through controlled acoustic transformations — pitch shifting, time stretching, and noise injection. These augmentations increase diversity and improve robustness while maintaining full reproducibility and minimal external dependencies.

The generator produces four functional categories of data:

- **hand grab** – wake-word class representing the grasp/closing command  
- **hand release** – complementary wake-word class for the opening command  
- **other words** – distractor phrases used to improve model generalization  
- **background** – synthetic noise segments used as negative examples  

All audio samples are normalized to **16 kHz**, **mono**, and a fixed **2-second duration**, ensuring compatibility with keyword-spotting architectures commonly used in embedded systems, TinyML frameworks, TensorFlow Lite, and EdgeImpulse pipelines.

This notebook serves as both a technical reference and a practical tool for generating custom datasets for speech-driven prosthetic-hand control.  
By running the cells, the full dataset will be automatically created inside the `dataset/` directory.


## Install and Setup

This section installs the required dependencies for running the synthetic dataset generator in Google Colab. The setup includes audio processing utilities and the MMS-TTS English model from HuggingFace, ensuring a reproducible and fully open-source environment without external API dependencies.

### Environment Setup

This section installs all required dependencies for running the synthetic voice-command dataset generator in Google Colab. The setup includes core audio-processing libraries and system utilities needed for synthesis, resampling, and waveform manipulation.


In [1]:
!apt-get -y install -qq ffmpeg
!pip install --quiet transformers librosa soundfile pydub numpy
print("✅ Environment ready.")


✅ Environment ready.


### Model Initialization

The following step loads the MMS-TTS English model through the HuggingFace `pipeline` interface. This lightweight text-to-speech model is fully open-source, runs reliably on CPU, and serves as the baseline synthesizer to generate the speech segments used in all dataset classes.


In [2]:
from transformers import pipeline

# Load small English MMS-TTS model (stable on CPU/Colab)
tts = pipeline(
    task="text-to-speech",
    model="facebook/mms-tts-eng",
    device="cpu"
)

print("✅ MMS-TTS model loaded successfully.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

Device set to use cpu


✅ MMS-TTS model loaded successfully.


### Base Speech Synthesis Function

The MMS-TTS English model produces different output structures depending on internal batching and execution conditions (e.g., dictionaries or lists containing dictionaries). To ensure consistent processing across all dataset classes, the following function standardizes the TTS output into a single 1D `float32` NumPy array. This normalization step is essential for downstream augmentation, padding, and fixed-length alignment, and guarantees reproducibility across executions and runtime environments.


In [3]:
import numpy as np

# Audio normalization parameters
TARGET_SR = 16000   # Sampling rate (Hz)
DURATION_S = 2.0    # Target duration (seconds)
DURATION_SAMPLES = int(TARGET_SR * DURATION_S)

def synthesize_base(text):
    """
    Synthesizes speech from the input text using the MMS-TTS model.
    Ensures consistent output format by converting the model output
    into a 1D float32 NumPy array. Handles dict and list-of-dicts
    output variants generated by the HuggingFace pipeline.

    Parameters
    ----------
    text : str
        Input text to synthesize.

    Returns
    -------
    np.ndarray
        1D float32 array containing audio samples.

    Raises
    ------
    ValueError
        If the model returns an unexpected structure or missing audio.
    """
    output = tts(text)

    # Normal case: dictionary output
    if isinstance(output, dict):
        y = output.get("audio", None)

    # Alternative case: list containing dictionary output
    elif isinstance(output, list) and len(output) > 0:
        item = output[0]
        if isinstance(item, dict):
            y = item.get("audio", None)
        else:
            raise ValueError(f"Invalid MMS-TTS list element type: {type(item)}")

    else:
        raise ValueError(f"Unsupported MMS-TTS output type: {type(output)}")

    if y is None:
        raise ValueError(f"No audio returned for text '{text}'.")

    # Ensure float32 NumPy array
    y = np.asarray(y, dtype=np.float32)

    # Flatten multi-dimensional output
    if y.ndim > 1:
        y = y.reshape(-1)

    return y

print("Base synthesis function successfully initialized.")


Base synthesis function successfully initialized.


### Data Augmentation Function

To increase intra-class and inter-speaker variability, the synthesized waveforms undergo controlled acoustic perturbations. These transformations simulate natural variability in pitch, speaking rate, and background conditions while preserving command intelligibility. The augmentation block applies three operations—pitch shifting (±4 semitones), time-stretching (±10%), and Gaussian noise injection—using conservative ranges to avoid destructive artifacts. Each transformation is applied stochastically, ensuring that every synthesized instance differs from the base waveform without compromising reproducibility. The function returns a normalized 1D `float32` array compatible with subsequent trimming and padding stages.


In [4]:
import librosa
import numpy as np

def augment_audio(y):
    """
    Applies controlled acoustic perturbations to simulate speaker and
    recording variability. The transformations include stochastic pitch
    shifting (±4 semitones), time-stretching (0.9–1.1×), and Gaussian-noise
    injection. The function ensures the output is always a valid 1D float32
    NumPy array suitable for downstream processing.

    Parameters
    ----------
    y : np.ndarray
        Input audio signal (1D array).

    Returns
    -------
    np.ndarray
        Augmented audio signal as a 1D float32 array.
    """
    # Ensure valid 1D float32 format
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    # Pitch shift (±4 semitones)
    try:
        if np.random.rand() < 0.5 and y.shape[0] > 512:
            steps = float(np.random.uniform(-4, 4))
            y = librosa.effects.pitch_shift(y, sr=TARGET_SR, n_steps=steps).astype(np.float32)
    except Exception:
        pass

    # Time stretching (0.9–1.1)
    try:
        if np.random.rand() < 0.5 and y.shape[0] > 1024:
            rate = float(np.random.uniform(0.9, 1.1))
            y = librosa.effects.time_stretch(y, rate)
            y = np.asarray(y, dtype=np.float32).reshape(-1)
    except Exception:
        pass

    # Additive Gaussian noise
    try:
        if np.random.rand() < 0.5:
            noise = np.random.normal(0, 0.02, size=y.shape).astype(np.float32)
            y = (y + noise).astype(np.float32)
    except Exception:
        pass

    # Safety fallback: ensure valid output
    if y is None or not isinstance(y, np.ndarray) or y.ndim != 1:
        y = np.zeros(DURATION_SAMPLES, dtype=np.float32)

    return y.astype(np.float32)



### Waveform Length Normalization

All synthesized and augmented signals must be aligned to a fixed duration to ensure compatibility with keyword-spotting architectures and embedded inference engines. The following function enforces a strict 2-second window by trimming excess samples or zero-padding shorter sequences. This process guarantees uniform input length across all dataset classes and prevents downstream inconsistencies during feature extraction or model training.


In [5]:
def normalize_length(y):
    """
    Enforces a fixed-length waveform by trimming or zero-padding
    the input signal to DURATION_SAMPLES. Ensures consistent sizing
    required for downstream feature extraction and model training.

    Parameters
    ----------
    y : array-like
        Input audio waveform.

    Returns
    -------
    np.ndarray
        1D float32 waveform of length DURATION_SAMPLES.
    """
    if y is None:
        raise ValueError("normalize_length received None")

    # Ensure contiguous 1D float32 array
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    # Pad if shorter than target duration
    if y.shape[0] < DURATION_SAMPLES:
        y = np.pad(y, (0, DURATION_SAMPLES - y.shape[0]), mode="constant")

    # Trim if longer
    else:
        y = y[:DURATION_SAMPLES]

    return y


### End-to-End Synthesis Pipeline

The following function integrates all processing stages required to generate a final waveform, including base TTS synthesis, stochastic data augmentation, duration normalization, and file export. Each step includes defensive checks to guarantee that no invalid or undefined signal propagates through the pipeline. This ensures deterministic behavior and prevents runtime failures during large-scale dataset generation. The function saves the resulting waveform as a 16 kHz mono WAV file and returns the output path.


In [6]:
import soundfile as sf
import numpy as np

def synthesize_and_save(text, out_path):
    """
    Executes the complete audio-generation pipeline:
      1) Text-to-speech synthesis.
      2) Acoustic augmentation.
      3) Fixed-length normalization.
      4) WAV export.

    All stages include robust fallback mechanisms to prevent
    propagation of invalid or undefined signals.

    Parameters
    ----------
    text : str
        Input command to be synthesized.
    out_path : str
        Destination path for the output WAV file.

    Returns
    -------
    str
        Path to the saved audio file.
    """

    # --- 1. Base synthesis ---
    try:
        y = synthesize_base(text)
    except Exception:
        # Fallback: silence frame
        y = np.zeros(DURATION_SAMPLES, dtype=np.float32)

    # Ensure array consistency
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    # --- 2. Augmentation stage ---
    try:
        y_aug = augment_audio(y)
        if isinstance(y_aug, np.ndarray) and y_aug.ndim == 1:
            y = y_aug
        else:
            # Fallback to unaugmented signal
            y = np.asarray(y, dtype=np.float32).reshape(-1)
    except Exception:
        y = np.asarray(y, dtype=np.float32).reshape(-1)

    # --- 3. Length normalization ---
    try:
        y = normalize_length(y)
    except Exception:
        y = np.zeros(DURATION_SAMPLES, dtype=np.float32)

    # --- 4. Export ---
    sf.write(out_path, y, TARGET_SR)

    return out_path



### Dataset Generation

This section executes the full dataset synthesis process. For each command category, the pipeline generates the specified number of samples using the previously defined synthesis, augmentation, and normalization functions. Class-specific text prompts are used for the target wake-words, while distractor phrases and synthetic noise are employed for non-target classes. All samples are saved in structured directories under `dataset/`, enabling direct use in training pipelines for keyword spotting and embedded speech-recognition tasks.


In [7]:
import os
import numpy as np
import soundfile as sf

# Dataset root directory
root = "dataset"

# Target class folders
classes = ["hand_grab", "hand_release", "background", "other_words"]

# Create class directories
for c in classes:
    os.makedirs(os.path.join(root, c), exist_ok=True)

print("Folder structure ready.")

# Number of samples per category
N_WAKE = 100     # hand grab & hand release
N_BG = 100       # background noise
N_OTHER = 100    # other words

# ------------------------------------
# Generate "hand grab" samples
# ------------------------------------
print("Generating 'hand grab' samples...")
for i in range(N_WAKE):
    out_path = f"{root}/hand_grab/hand_grab_{i:04d}.wav"
    synthesize_and_save("hand grab", out_path)
print(f"{N_WAKE} samples generated for 'hand grab'.")

# ------------------------------------
# Generate "hand release" samples
# ------------------------------------
print("Generating 'hand release' samples...")
for i in range(N_WAKE):
    out_path = f"{root}/hand_release/hand_release_{i:04d}.wav"
    synthesize_and_save("hand release", out_path)
print(f"{N_WAKE} samples generated for 'hand release'.")

# ------------------------------------
# Generate "background" samples
# ------------------------------------
print("Generating 'background' samples...")

for i in range(N_BG):
    # Synthetic noise baseline
    noise = np.random.normal(0, 0.02, size=DURATION_SAMPLES).astype(np.float32)
    out_path = f"{root}/background/bg_{i:04d}.wav"
    sf.write(out_path, noise, TARGET_SR)

print(f"{N_BG} background samples generated.")

# ------------------------------------
# Generate "other words" samples
# ------------------------------------
print("Generating 'other words' samples...")

other_phrases = [
    "hello", "go", "stop", "open hand", "close hand",
    "move forward", "grab it", "release it", "good job", "okay"
]

for i in range(N_OTHER):
    phrase = np.random.choice(other_phrases)
    out_path = f"{root}/other_words/other_{i:04d}.wav"
    synthesize_and_save(phrase, out_path)

print(f"{N_OTHER} 'other words' samples generated.")

print("\nDataset generation complete. Files saved in 'dataset/' directory.")



Folder structure ready.
Generating 'hand grab' samples...
100 samples generated for 'hand grab'.
Generating 'hand release' samples...
100 samples generated for 'hand release'.
Generating 'background' samples...
100 background samples generated.
Generating 'other words' samples...


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1280
  warnings.warn(


100 'other words' samples generated.

Dataset generation complete. Files saved in 'dataset/' directory.


## Optional: Download Dataset as ZIP

If you want to export the generated dataset for local training, offline experiments, or integration into another environment, you can use the following cell.  
It automatically compresses the entire `dataset/` directory into a ZIP file and triggers a direct download through Google Colab.

⚠️ **Note:** Depending on the dataset size, the download may take several seconds to prepare.


In [8]:
import shutil
from google.colab import files
import os

# Ensure the folder exists
if os.path.exists("dataset"):
    # Create ZIP archive
    shutil.make_archive("dataset", "zip", "dataset")
    print("✅ dataset.zip created successfully.")

    # Trigger download
    files.download("dataset.zip")
else:
    print("❌ Directory 'dataset' not found. Please generate the dataset first.")


✅ dataset.zip created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Notes on Reproducibility

All generated samples are deterministic given identical random seeds and execution order.
If strict reproducibility is required for experimental comparison, uncomment and set a fixed seed at the beginning of the notebook:

```python
# np.random.seed(42)


### Citation

If you use this notebook, its code, or the generated synthetic dataset in your research, please cite the associated work [TO DETERMINE]

The complete implementation and updated versions of this notebook are maintained in the accompanying GitHub repository:

**https://github.com/ximocerda/ProstheticHand-VoiceCommands**
